In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
# base
df = pd.read_csv("../data/demanda_supermercado_2023_2024.csv")
df["data"] = pd.to_datetime(df["data"])
df["ano"] = df["data"].dt.year
df["mes"] = df["data"].dt.month
df["dia_semana"] = df["data"].dt.day_name()

In [3]:
# Features e target
features = ["produto", "categoria", "mes", "dia_semana", "preco_unitario"]
target = "quantidade_vendida"
X = df[features]
y = df[target]

In [4]:
# Separação treino e teste por ano
X_train = X[df["ano"] == 2023]
X_test = X[df["ano"] == 2024]
y_train = y[df["ano"] == 2023]
y_test = y[df["ano"] == 2024]

In [5]:
# Pipeline com OrdinalEncoder e GradientBoosting
preprocessor = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), ["produto", "categoria", "dia_semana"])
], remainder="passthrough")

pipeline_gbr = Pipeline([
    ("preprocess", preprocessor),
    ("model", GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42))
])

In [6]:
# Treino modelo
pipeline_gbr.fit(X_train, y_train)

C:\Users\jessi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['produto', 'categoria',
                                                   'dia_semana'])])),
                ('model', GradientBoostingRegressor(random_state=42))])

In [11]:
# Função para prever demanda em um dia específico
def prever_demanda_para_data(data_previsao: str, modelo, salvar_csv=True):
    data_ref = pd.to_datetime(data_previsao)
    produtos_unicos = df[["produto", "categoria"]].drop_duplicates().reset_index(drop=True)
    base_previsao = produtos_unicos.copy()
    base_previsao["mes"] = data_ref.month
    base_previsao["dia_semana"] = data_ref.day_name()
    precos_medio = df.groupby("produto")["preco_unitario"].mean().reset_index()
    base_previsao = base_previsao.merge(precos_medio, on="produto", how="left")
    previsoes = modelo.predict(base_previsao)
    resultado = base_previsao.copy()
    resultado["data_previsao"] = data_ref.date()
    resultado["previsao"] = np.round(previsoes).astype(int)
    
    df_resultado = resultado[["data_previsao", "produto", "categoria", "mes", "dia_semana", "preco_unitario", "previsao"]]
    
    if salvar_csv:
        caminho_arquivo = f"../data/previsao_demanda_{data_previsao}.csv".replace(":", "-")
        df_resultado.to_csv(caminho_arquivo, index=False)
        return caminho_arquivo
    
    return df_resultado

# Testa salvamento para o dia 2024-01-02
prever_demanda_para_data("2024-01-02", pipeline_gbr, salvar_csv=True)


'../data/previsao_demanda_2024-01-02.csv'

In [13]:
# Definir limites por categoria
limites_categoria = {
    "Hortifruti": 80,
    "Bebidas": 120,
    "Mercearia": 100,
    "Padaria": 70,
    "Laticínios": 90,
    "Higiene": 60,
    "Limpeza": 60,
    "Carnes": 130
}

# Aplicar a lógica de alerta na previsão de 2024-01-02
df_previsao["limite_reposicao"] = df_previsao["categoria"].map(limites_categoria)
df_previsao["alerta_reposicao"] = df_previsao["previsao"] > df_previsao["limite_reposicao"]

# Exportar novo CSV com alertas
caminho_alerta = "../data/previsao_alerta_2024-01-02.csv"
df_previsao.to_csv(caminho_alerta, index=False)
caminho_alerta

'../data/previsao_alerta_2024-01-02.csv'